
# 第 3 章 线性模型与概率编程语言
***

随着概率编程语言的出现,现代贝叶斯建模有时候可以简单到"写好模型、按下运行按钮"就完事。但要
做好模型搭建与分析,通常还需要更多的工作。本书接下来会搭建许多不同类型的模型,而本章我们先
从最朴素的线性模型开始。线性模型是一大类模型的统称,它们的共同点是:观测值的期望是相关预测
变量的线性组合。牢固掌握线性模型的拟合与解读方法,是理解后续更复杂模型的坚实基础,也有助于
巩固我们在第 1、2 章学到的贝叶斯推断与探索性分析基础,并把它们应用到不同的概率编程语言上。

本章会介绍本书接下来大部分篇幅都会用到的两种概率编程语言(PPL):你已经初步接触过的 PyMC,
以及 TensorFlow Probability(TFP)。在用这两种 PPL 搭建模型的同时,我们会重点关注同一套统计
思想在不同 PPL 里是如何映射到具体实现的。我们会先拟合一个"只有截距"(没有协变量)的模型,
再逐步加入一个或多个协变量,最终扩展到广义线性模型。读完本章,你会对线性模型更熟悉,对贝叶斯
工作流的许多步骤更了解,也会更适应用 PyMC、TFP 和 ArviZ 三者结合来完成贝叶斯工作流。

**关于本章数据集**:本章使用 Palmer Penguins(帕默企鹅)数据集,已随本仓库一同提供
(`data/penguins.csv`),无需额外下载。

## 比较两个(或更多)组

要说到"比较",很难有什么比企鹅更适合的对象了——这些可爱又不会飞的鸟儿谁能不喜欢呢?我们
可能想问:"每种企鹅的平均体重是多少?"或者"这些平均值之间差异有多大?",用统计学的话说就是
"平均值的离散程度是多少?"研究者 Kristen Gorman 恰好也很喜欢研究企鹅,她走访了南极的 3 座
岛屿,采集了 Adelie(阿德利)、Gentoo(巴布亚)、Chinstrap(帽带)三种企鹅的数据,汇总成了
Palmer Penguins 数据集。观测项目包括企鹅的体重、鳍肢长度、性别等身体特征,以及它们所在岛屿等
地理特征。

我们先加载数据,并过滤掉存在缺失值的行——这被称为"完整案例分析"(complete case
analysis),顾名思义,就是只使用所有观测项都齐全的行。虽然也可以用数据插补(data
imputation)等方式来处理缺失值,但本章为简单起见,采用最简单直接的做法。


In [ ]:
import pymc as pm
import matplotlib.pyplot as plt
import arviz as az
import xarray as xr
import pandas as pd
from scipy import special, stats
import numpy as np


In [ ]:
az.style.use("arviz-grayscale")
plt.rcParams['figure.dpi'] = 300


In [ ]:
penguins = pd.read_csv("data/penguins.csv")

# 只保留后面会用到的列
missing_data = penguins.isnull()[
    ["bill_length_mm", "flipper_length_mm", "sex", "body_mass_g"]
].any(axis=1)

# 丢弃存在任意缺失值的行
penguins = penguins.loc[~missing_data]

penguins.head()


In [ ]:
penguins.shape



接下来我们只用几行代码,就能算出体重(`body_mass_g`)的经验均值和标准差,按物种分组汇总。


In [ ]:
summary_stats = (penguins.loc[:, ["species", "body_mass_g"]]
                         .groupby("species")
                         .agg(["mean", "std", "count"]))
summary_stats



现在我们有了均值和离散程度的点估计,但并不知道这些统计量本身的不确定性。获得不确定性估计的
一种方法就是使用贝叶斯方法。为此我们需要为"观测值与参数之间的关系"提出一个猜想,例如:

$$
\overbrace{p(\mu, \sigma \mid Y)}^{\text{后验}} \propto \overbrace{\mathcal{N}(Y \mid \mu, \sigma)}^{\text{似然}}\;  \overbrace{\underbrace{\mathcal{N}(4000, 3000)}_{\mu}
     \underbrace{\mathcal{H}\text{T}(100, 2000)}_{\sigma}}^{\text{先验}}
$$

这个公式其实就是把"后验 正比于 似然乘以先验"这个通用公式,针对每个具体参数明确写了出来。
由于我们没有特别的理由选用信息量很大的先验,这里对 $\mu$ 和 $\sigma$ 都采用宽泛的先验,并且
先验的选择本身也参考了观测数据的经验均值和标准差。此外,我们先不去估计所有物种的体重,而是
先从 Adelie 企鹅的体重入手。高斯分布是企鹅体重(以及一般生物体重)似然的合理选择,我们就
采用它。下面把上式翻译成一个计算模型:


In [ ]:
adelie_mask = (penguins["species"] == "Adelie")
adelie_mass_obs = penguins.loc[adelie_mask, "body_mass_g"].values

with pm.Model() as model_adelie_penguin_mass:
    σ = pm.HalfStudentT("σ", 100, 2000)
    μ = pm.Normal("μ", 4000, 3000)
    mass = pm.Normal("mass", mu=μ, sigma=σ, observed=adelie_mass_obs)

    idata_adelie_mass = pm.sample(chains=4)
    idata_adelie_mass.extend(pm.sample_prior_predictive(samples=5000))



在计算后验之前,我们先检查一下先验。具体来说,首先要确认从模型采样在计算上是可行的,其次要
确认基于领域知识,我们选择的先验是合理的。下图画出了先验样本。既然能画出图来,说明模型本身
没有明显的计算问题,比如形状不匹配或者随机变量/似然设置错误。从先验样本本身可以看出,我们
对企鹅体重的可能取值并没有过度约束,甚至可能约束得还不够——因为体重均值的先验里居然包含了
负值。不过考虑到这只是一个简单模型,而且我们有足够多的观测数据,这里先记下这个小瑕疵,继续
往下估计后验分布。

拟合完成后,我们可以画出 KDE 和秩图(见下下图,共 4 个子图,右边两个是秩图,左边是每个参数
每条链的 KDE),再结合数值诊断结果,确认链已经收敛。运用第 2 章建立的直觉,可以判断这次拟合
是可接受的,于是我们继续分析。


In [ ]:
axes = az.plot_posterior(idata_adelie_mass.prior, var_names=["σ", "μ"], figsize=(10, 4))

plt.savefig("img/chp03/single_species_prior_predictive.png")


In [ ]:
axes = az.plot_trace(idata_adelie_mass, divergences="bottom", kind="rank_bars", figsize=(10, 4))

plt.savefig("img/chp03/single_species_KDE_rankplot.png")


In [ ]:
az.summary(idata_adelie_mass)



拟合结果令人满意后,我们把所有链合并画一张后验图,并与前面的经验均值/标准差点估计对照来看
(下图中的竖直虚线就是经验均值和标准差)。


In [ ]:
axes = az.plot_posterior(idata_adelie_mass, hdi_prob=.94, figsize=(10, 4));

# 数值取自前面 az.summary 的输出
axes[0].axvline(3706, linestyle="--")
axes[1].axvline(459, linestyle="--")
plt.savefig("img/chp03/single_species_mass_posteriorplot.png")



用贝叶斯方法估计,我们得到的不只是一个点,而是一整个"合理参数"的分布。结合数值汇总表,均值
落在 3632 到 3772 克之间都是相当合理的。注意标准差的边缘后验分布本身的取值范围也不小。还要
记住:这个后验分布并不是"某一只企鹅体重"的分布,而是我们假设描述企鹅体重的那个高斯分布,其
参数的可能取值分布。如果想要单只企鹅体重的估计分布,需要生成后验预测分布——在这个例子中,
那会是以 $\mu$、$\sigma$ 的后验为条件的、同一个高斯分布。

搞清楚了 Adelie 企鹅的体重之后,我们可以用同样的方法处理另外两个物种。当然可以再写两个模型,
但更简便的做法是:直接写一个模型,同时处理 3 个分组(每个物种一组)。


In [ ]:
# pd.Categorical 能让我们方便地在下面按物种做索引
all_species = pd.Categorical(penguins["species"])
coords = {"species": all_species.categories}

with pm.Model(coords=coords) as model_penguin_mass_all_species:
    # 注意这里加上了 dims 参数(等价于原书 PyMC3 里的 shape=3),
    # 让每个物种都拥有各自独立的 σ、μ
    σ = pm.HalfStudentT("σ", 100, 2000, dims="species")
    μ = pm.Normal("μ", 4000, 3000, dims="species")
    mass = pm.Normal("mass",
                     mu=μ[all_species.codes],
                     sigma=σ[all_species.codes],
                     observed=penguins["body_mass_g"])

    idata_penguin_mass_all_species = pm.sample()



我们利用了每个参数上可选的 `dims`(在旧版 PyMC3 中是 `shape`)参数,并在似然里加上索引,
告诉 PyMC 我们希望针对每个物种分别估计后验。在编程语言设计中,这种让表达想法更顺手的小技巧
被称为**语法糖(syntactic sugar)**,概率编程语言的开发者们同样也提供了不少这样的语法糖——
概率编程语言致力于让表达模型更轻松、更不容易出错。

拟合之后我们再检查一下 KDE 和秩图。和只有 Adelie 一个物种时的图相比,现在多出了 4 个子图
(对应新增的两组参数)。可以花点时间,把每个物种的均值估计与前面各物种的经验均值对照一下。
为了更方便比较各物种之间分布的差异,我们再用森林图(forest plot)把后验画一遍,可以更清楚
地看出巴布亚(Gentoo)企鹅的体重似乎比阿德利(Adelie)或帽带(Chinstrap)企鹅更重。


In [ ]:
axes = az.plot_trace(idata_penguin_mass_all_species, compact=False,
                     divergences="bottom", kind="rank_bars", figsize=(10, 10));

plt.savefig("img/chp03/all_species_KDE_rankplot.png")


In [ ]:
axes = az.plot_forest(idata_penguin_mass_all_species, var_names=["μ"], figsize=(8, 2.5))
axes[0].set_title("μ Mass Estimate: 94.0% HDI")

plt.savefig("img/chp03/independent_model_forestplotmeans.png")



再来看看标准差的森林图。后验的 94% 最高密度区间显示,不确定性大致在 100 克这个量级。


In [ ]:
axes = az.plot_forest(idata_penguin_mass_all_species, var_names=["σ"], figsize=(8, 3));
axes[0].set_title("σ Mass Estimate: 94.0% HDI")

plt.savefig("img/chp03/independent_model_forestplotsigma.png")



### 比较两种 PPL

在继续深入统计与建模思想之前,我们先花点时间聊聊概率编程语言本身,并介绍本书会用到的另一种
PPL——TensorFlow Probability(TFP)。我们会把上面刚写好的、只有截距的 PyMC 模型,翻译成
TFP 版本。

你可能会觉得"没必要学好几种 PPL 吧",但本书选择用两种(而不是一种)PPL,是有具体原因的。
用不同 PPL 实现同一套工作流,能让你对计算贝叶斯建模的理解更全面,帮助你把"计算细节"和
"统计思想"区分开来,让你成为更强的建模者。此外,不同 PPL 各有侧重和优势:PyMC 是更高层的
PPL,用更少的代码就能表达模型;而 TFP 提供的是更底层、可组合的建模与推断原语。还有一点是,
并非所有 PPL 都能同样轻松地表达所有模型——比如时间序列模型(第 6 章)用 TFP 表达更顺手,而
贝叶斯加法回归树(第 7 章)用 PyMC 表达更顺手。通过接触多种语言,你会对贝叶斯建模的基本要素
及其计算实现都有更扎实的理解。

概率编程语言(重点是"语言"二字)由一系列**原语(primitives)**组成。编程语言中的原语,是
用来构建更复杂程序的最基本元素——可以把原语类比成自然语言里的单词,能组合成更复杂的结构
(比如句子)。就像不同的自然语言用不同的单词一样,不同的 PPL 也使用不同的原语,主要用来表达
模型、执行推断,或完成工作流的其他部分。在 PyMC 中,与建模相关的原语都在 `pm.` 命名空间下,
比如上面模型里的 `pm.HalfStudentT(.)`、`pm.Normal(.)`,它们代表随机变量。`with pm.Model()
as .` 语句会启用一个 Python 上下文管理器,PyMC 借助它收集上下文管理器内部的随机变量,构建出
模型。随后我们分别用 `pm.sample_prior_predictive(.)` 和 `pm.sample(.)`,从先验预测分布和
后验分布中获得样本。

类似地,TFP 也在 `tfp.distributions` 中提供了指定分布和模型的原语,在 `tfp.mcmc` 中提供了
运行 MCMC 的原语,等等。比如,要搭建一个贝叶斯模型,TensorFlow 在 `tfd.JointDistribution`
这一 API 下提供了多种原语。本章及全书后续大多使用 `tfd.JointDistributionCoroutine`,不过
`tfd.JointDistribution` 还有其他变体,可能更适合你的具体场景 [^1]。由于基础的数据加载和
汇总统计和前面完全一样,我们可以把注意力集中在建模和推断本身。下面是用 TFP 表达的、和前面
`model_penguin_mass_all_species` 完全等价的模型:


In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp

tfd = tfp.distributions
root = tfd.JointDistributionCoroutine.Root

species_idx = tf.constant(all_species.codes, tf.int32)
body_mass_g = tf.constant(penguins["body_mass_g"], tf.float32)

@tfd.JointDistributionCoroutine
def jd_penguin_mass_all_species():
    σ = yield root(tfd.Sample(
            tfd.HalfStudentT(df=100, loc=0, scale=2000),
            sample_shape=3,
            name="sigma"))
    μ = yield root(tfd.Sample(
            tfd.Normal(loc=4000, scale=3000),
            sample_shape=3,
            name="mu"))
    mass = yield tfd.Independent(
        tfd.Normal(loc=tf.gather(μ, species_idx, axis=-1),
                   scale=tf.gather(σ, species_idx, axis=-1)),
        reinterpreted_batch_ndims=1,
        name="mass")



既然这是我们第一次接触用 TFP 写的贝叶斯模型,不妨多花几段篇幅讲讲这套 API。TFP 的原语是
`tfp.distributions` 里的分布类,我们给它取了个更短的别名 `tfd = tfp.distributions`。`tfd`
包含了常用分布,比如 `tfd.Normal(.)`。我们还用到了 `tfd.Sample`,它会返回基础分布的多个
独立副本(概念上类似于 PyMC 里 `shape=(.)` 这个语法糖起到的作用)。`tfd.Independent` 用来
表明:这个分布包含多个副本,我们希望在计算对数似然时对某个轴求和,具体是哪个轴由
`reinterpreted_batch_ndims` 参数指定。通常我们会用 `tfd.Independent` 包裹与观测值相关的
分布 [^2]。

`tfd.JointDistributionCoroutine` 模型有一个有意思的特征——顾名思义,就是用到了 Python 里的
协程(coroutine)。不深入讲生成器和协程的细节,这里你只需要知道:对一个分布使用 `yield`
语句,会在模型函数内部给你一个随机变量。可以把 `y = yield Normal(.)` 理解为在表达
$y \sim \text{Normal(.)}$。此外,我们需要用 `tfd.JointDistributionCoroutine.Root` 把没有
依赖关系的随机变量包裹起来,标记为"根节点"。整个模型被写成一个不接受输入参数、也没有返回值
的 Python 函数。最后,把 `@tfd.JointDistributionCoroutine` 作为装饰器加在函数上面,就能
直接得到模型对象(也就是一个 `tfd.JointDistribution`)。

得到的 `jd_penguin_mass_all_species`,就是前面那个"只有截距"回归模型的 TFP 版本。它和其他
`tfd.Distribution` 一样拥有类似的方法,可以在贝叶斯工作流中使用。比如要抽取先验和先验预测
样本,可以调用 `.sample(.)` 方法,它会返回一个类似 `namedtuple` 的自定义嵌套 Python 结构。
下面这行代码抽取了 1000 个先验及先验预测样本:


In [ ]:
prior_predictive_samples = jd_penguin_mass_all_species.sample(1000)



`tfd.JointDistribution` 的 `.sample(.)` 方法也可以抽取条件样本,这正是我们后面用来抽取后验
预测样本的机制。你可以运行下面这段代码,观察当我们把模型中某些随机变量条件化为特定值时,随机
样本会如何变化。总的来说,调用 `.sample(.)` 时,我们启动的是模型的*正向*生成过程。


In [ ]:
jd_penguin_mass_all_species.sample(sigma=tf.constant([.1, .2, .3]))
jd_penguin_mass_all_species.sample(mu=tf.constant([.1, .2, .3]));



一旦我们把生成模型 `jd_penguin_mass_all_species` 条件化到观测到的企鹅体重上,就能得到后验
分布。从计算的角度看,我们希望构造一个函数,在给定输入时返回后验对数概率(不计常数项)。这
可以通过创建一个 Python 函数闭包,或者使用 `.experimental_pin` 方法来实现:


In [ ]:
target_density_function = lambda *x: jd_penguin_mass_all_species.log_prob(*x, mass=body_mass_g)

jd_penguin_mass_observed = jd_penguin_mass_all_species.experimental_pin(mass=body_mass_g)
target_density_function = jd_penguin_mass_observed.unnormalized_log_prob



有了 `target_density_function`,就可以进行推断了——比如可以求这个函数的最大值,得到**最大
后验概率(MAP)估计**;也可以用 `tfp.mcmc` 里的方法从后验中采样。或者更方便地,使用一套和
PyMC 里类似的标准采样流程,如下所示:


In [ ]:
run_mcmc = tf.function(
    tfp.experimental.mcmc.windowed_adaptive_nuts,
    autograph=False, jit_compile=True)
mcmc_samples, sampler_stats = run_mcmc(
    1000, jd_penguin_mass_all_species, n_chains=4, num_adaptation_steps=1000,
    mass=body_mass_g)

idata_penguin_mass_all_species2 = az.from_dict(
    posterior={
        # TFP 的 mcmc 结果形状是 (num_samples, num_chains, ...),
        # 我们交换第一、第二个轴,让形状符合 ArviZ 的预期
        k:np.swapaxes(v, 1, 0)
        for k, v in mcmc_samples._asdict().items()},
    sample_stats={
        k:np.swapaxes(sampler_stats[k], 1, 0)
        for k in ["target_log_prob", "diverging", "accept_ratio", "n_steps"]}
)


In [ ]:
az.plot_trace(idata_penguin_mass_all_species2, divergences="bottom", kind="rank_bars", figsize=(10,4));



上面这段代码运行了 4 条 MCMC 链,每条链在 1000 步适应(adaptation)之后,采集 1000 个后验
样本。内部实现是通过 `experimental_pin` 方法,把观测值(以关键字参数 `mass=body_mass_g` 的
形式)条件化进模型。随后代码把采样结果解析成 ArviZ 的 `InferenceData`,这样我们就可以用
ArviZ 来做诊断和探索性分析了。我们还可以进一步、以同样透明的方式,把先验样本、后验预测样本
和数据对数似然添加进 `idata_penguin_mass_all_species2`。注意这里用到了
`tfd.JointDistribution` 的 `sample_distributions` 方法,它既能抽样,又能生成一个以后验样本
为条件的分布对象。


In [ ]:
prior_predictive_samples = jd_penguin_mass_all_species.sample([1, 1000])
dist, samples = jd_penguin_mass_all_species.sample_distributions(
    value=mcmc_samples)
ppc_samples = samples[-1]
ppc_distribution = dist[-1].distribution
data_log_likelihood = ppc_distribution.log_prob(body_mass_g)

# 注意不要在同一个 REPL 会话里重复运行这段代码(add_groups 不能重复添加同名的组)
idata_penguin_mass_all_species2.add_groups(
    prior=prior_predictive_samples[:-1]._asdict(),
    prior_predictive={"mass": prior_predictive_samples[-1]},
    posterior_predictive={"mass": np.swapaxes(ppc_samples, 1, 0)},
    log_likelihood={"mass": np.swapaxes(data_log_likelihood, 1, 0)},
    observed_data={"mass": body_mass_g}
)


In [ ]:
az.plot_ppc(idata_penguin_mass_all_species2, num_pp_samples=50, figsize=(10, 3));


In [ ]:
az.loo(idata_penguin_mass_all_species2)



到这里,我们就快速游览完了 TensorFlow Probability。和学习任何语言一样,第一次接触不太可能
立刻精通。但通过对比这两个模型,你现在应该已经对"哪些概念是贝叶斯建模本身的核心",哪些概念
"只是某个 PPL 特有的表达方式"有了更清晰的认识。本章剩余部分以及下一章,我们会在 PyMC 和 TFP
之间切换,继续帮助你辨析这种区别,并展示更多实战例子。我们也准备了一些练习,让你把某个语言
写的示例代码翻译成另一种语言,帮助你在成为"PPL 多语者"的路上继续练习。

## 线性回归

前面一节,我们通过对高斯分布的均值和标准差设置先验,为企鹅体重的分布建模,并且假设体重不随
数据中的其他特征而变化。但直觉上,其他观测到的数据点应该能为企鹅体重提供信息:如果我们看到
两只企鹅,一只鳍肢很长,一只鳍肢很短,我们会预期鳍肢更长的那只体重更大——即便手头没有秤,也
能这样推测。要估计"观测到的鳍肢长度"与"体重估计值"之间的这种关系,最简单的方法之一,就是
拟合一个线性回归模型:把均值*条件性地*建模为其他变量的线性组合:

$$
\begin{split}
    \mu =& \beta_0 + \beta_1 X_1 + \dots + \beta_m X_m \\
Y \sim& \mathcal{N}(\mu, \sigma)
\end{split}
$$

其中系数(也叫参数)用 $\beta_i$ 表示,比如 $\beta_0$ 是线性模型的截距。$X_i$ 通常被称为
预测变量或自变量,$Y$ 通常被称为目标、输出、响应或因变量。需要注意的是,$\boldsymbol{X}$
和 $Y$ 都是观测数据,而且是成对出现的 $\{y_j, x_j\}$——也就是说,如果只打乱 $Y$ 的顺序而不
同步打乱 $X$,就会破坏数据中的部分信息。

我们之所以称之为"线性"回归,是因为参数(而不是协变量本身)以线性方式进入模型。对只有一个
协变量的模型,可以把它想成是给 $(X, y)$ 数据拟合一条直线;维度更高时则是一个平面,或者更
一般地说是一个超平面。

我们也可以用矩阵记号改写上式:

$$
\mu = \mathbf{X}\boldsymbol{\beta}
$$

这里是系数列向量 $\beta$ 与协变量矩阵 $\mathbf{X}$ 的矩阵-向量乘积。

另一种你可能在(非贝叶斯)场合见过的写法,是把线性回归写成对某个线性预测的"带噪声观测":

$$
Y = \mathbf{X}\boldsymbol{\beta} + \epsilon,\; \epsilon \sim \mathcal{N}(0, \sigma)
$$

这种写法把线性回归的确定性部分(线性预测)和随机部分(噪声)分开了。不过我们更偏好前一种
写法,因为它能更清楚地展示生成过程。

> **设计矩阵(Design Matrix)**:矩阵 $\mathbf{X}$ 被称为设计矩阵,它是一组给定对象的解释
> 变量取值矩阵,再加上一列全为 1 的向量来表示截距。每一行代表一个独立的观测(比如一只企鹅),
> 后续每一列对应一个变量(比如鳍肢长度)在该对象上的具体取值。
>
> 设计矩阵不仅限于连续协变量。对于表示类别预测变量的离散协变量(即只有少数几个类别),一种
> 常见的转换成设计矩阵的方法叫做**哑变量编码(dummy coding)**或**独热编码(one-hot
> coding)**。比如在前面"每个物种一个截距"的模型里,除了用 `mu = μ[species.codes]`,我们
> 也可以用 `pandas.get_dummies` 把类别信息解析成设计矩阵,写成
> `mu = pd.get_dummies(penguins["species"]) @ μ`(`@` 是 Python 里做矩阵乘法的运算符)。
> Python 里还有其他做独热编码的工具,比如 `sklearn.preprocessing.OneHotEncoder`,毕竟这是
> 一种非常常见的数据处理技巧。
>
> 类别型预测变量也可以用另一种编码方式,让生成的列及其系数代表"线性对比"。比如在 ANOVA 的
> 零假设检验场景下,两个类别预测变量的不同设计矩阵编码方式,分别对应 I、II、III 型平方和。

如果把上面的公式画成"三维"图像,就会得到下图——它展示了似然分布的估计参数,是如何随着其他
观测数据 $x$ 变化的。虽然在本图以及本章中,我们都用线性关系来建模 $x$ 与 $Y$ 之间的关系、
并用高斯分布作为似然,但在其他模型架构里,我们也可能选用不同的方式,第 4 章会看到这一点。


In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(projection='3d')

x = np.linspace(-16, 12, 500)
z = np.array([0, 4, 8])

for i, zi in enumerate(z):
    dens = stats.norm(-zi, 3).pdf(x)
    ax.plot(x, dens, zs=zi + 1, zdir="y", c="k")
    ax.plot([-zi, -zi], [0, max(dens)], zs=zi + 1, c="k", ls=":", zdir="y")
    ax.text(
        -zi,
        zi - 1,
        max(dens) * 1.03 + i / 100,
        f"$\\mathcal{{N}}(\\beta_0 + \\beta_1 x_{i}, \\sigma)$",
        zdir="y",
        fontsize=18,
    )

ax.plot(-z, z + 1, "C4-", lw=3)


ax.set_xlabel("y", fontsize=20)
ax.set_ylabel("x", fontsize=24, labelpad=20)

ax.set_yticks([zi + 1 for zi in z])
ax.set_yticklabels([f"$x_{i}$" for i in range(len(z))], fontsize=22)
ax.grid(False)
ax.set_xticks([])
ax.set_zticks([])
ax.yaxis.pane.fill = False
ax.xaxis.pane.fill = False
ax.xaxis.pane.set_edgecolor("None")
ax.yaxis.pane.set_edgecolor("None")
ax.zaxis.pane.set_facecolor("C3")
ax.zaxis.line.set_linewidth(0)
ax.view_init(elev=10, azim=-25)
plt.savefig("img/chp03/3d_linear_regression.png", bbox_inches="tight", dpi=300)



### 线性回归里的企鹅

回到企鹅的例子,我们想用额外的数据来更好地估计一组企鹅的平均体重。用线性回归写出的模型如下,
包含两个新参数 $\beta_0$、$\beta_1$,通常分别称为截距和斜率。这里我们仍然设置宽泛的先验
$\mathcal{N}(0, 4000)$,把重点放在模型本身(这也等于说我们假设自己没有相关领域知识)。运行
采样器之后,我们现在估计了 $\sigma$、$\beta_1$、$\beta_0$ 三个参数。


In [ ]:
adelie_flipper_length_obs = penguins.loc[adelie_mask, "flipper_length_mm"]

with pm.Model() as model_adelie_flipper_regression:
    # pm.Data 能让我们在后面的代码块里改变这个变量的底层取值
    # PyMC 5.x 起 pm.MutableData 已合并进 pm.Data(默认就是可变的)
    adelie_flipper_length = pm.Data("adelie_flipper_length", adelie_flipper_length_obs)
    σ = pm.HalfStudentT("σ", 100, 2000)
    β_0 = pm.Normal("β_0", 0, 4000)
    β_1 = pm.Normal("β_1", 0, 4000)
    μ = pm.Deterministic("μ", β_0 + β_1 * adelie_flipper_length)

    mass = pm.Normal("mass", mu=μ, sigma=σ, observed=adelie_mass_obs)

    idata_adelie_flipper_regression = pm.sample()



为了节省篇幅,后面我们不会每次都展示诊断结果,但你不应该盲目信任我们(或你自己的采样器)——
请务必自己运行诊断,验证你得到的后验近似是可靠的。

采样完成后,我们画出下图,用来检视 $\beta_0$ 和 $\beta_1$ 的完整后验分布。系数 $\beta_1$
告诉我们:Adelie 企鹅鳍肢长度每变化 1 毫米,体重大致会变化 32 克,但在 22 到 41 克之间波动
也是合理的。此外还能看到,94% 最高密度区间并不跨越 0——这支持了"体重与鳍肢长度存在关系"这
一假设,对理解鳍肢长度与体重的相关性很有帮助。但我们也要小心,不要过度解读系数,或者误以为
线性模型必然意味着因果关系。比如,如果给一只企鹅做"延长鳍肢手术",并不必然会让它体重增加——
实际上由于应激或者觅食受阻,体重可能反而下降。反过来同样不一定成立:给企鹅提供更多食物,可能
帮助它长出更长的鳍肢,但也可能只是让它变得更肥胖。再看 $\beta_0$,它代表什么?根据后验估计,
如果我们见到一只鳍肢长度为 0 毫米的 Adelie 企鹅(当然这不可能存在),模型认为它的体重大概会
在 -4151 到 -510 克之间。按照模型来说这个陈述是"成立"的,但负数体重显然没有意义。这倒不一定
是个问题——并没有规定说模型里每个参数都必须可解释,也没有规定模型在每个参数取值处都要给出
合理的预测。就我们目前的目标而言,这个特定模型的目的是估计鳍肢长度和企鹅体重之间的关系,而
从后验估计来看,我们已经达成了这个目标。

> **模型:数学与现实之间的平衡**:在企鹅例子里,即便模型允许体重取值低于 0(甚至接近 0),
> 这样的取值也没有实际意义。因为我们是用远离 0 的体重值来拟合模型的,所以如果想把结论外推到
> 接近 0 或低于 0 的取值,模型失效也不足为奇。一个模型不一定需要对所有可能取值都给出合理的
> 预测,它只需要在我们搭建它的目的范围内,给出合理的预测就够了。

本节一开始我们猜测,加入协变量应该能让企鹅体重的预测更准确。我们可以通过比较"固定均值模型"
与"线性变化均值模型"给出的 $\sigma$ 后验估计来验证这一点(见下图):似然标准差的估计均值,从
约 460 克降到了约 380 克。


In [ ]:
axes = az.plot_posterior(idata_adelie_flipper_regression, var_names = ["β_0", "β_1"], figsize=(10, 3));

plt.savefig("img/chp03/adelie_coefficient_posterior_plots")


In [ ]:
axes = az.plot_forest(
    [idata_adelie_mass, idata_adelie_flipper_regression],
    model_names=["mass_only", "flipper_regression"],
    var_names=["σ"], combined=True, figsize=(10, 3.5))

axes[0].set_title("σ Comparison 94.0 HDI")
plt.savefig("img/chp03/SingleSpecies_SingleRegression_Forest_Sigma_Comparison.png")


In [ ]:
idata_adelie_flipper_regression.posterior["β_0"].mean().item()


In [ ]:
fig, ax = plt.subplots()
alpha_m = idata_adelie_flipper_regression.posterior["β_0"].mean().item()
beta_m = idata_adelie_flipper_regression.posterior["β_1"].mean().item()

flipper_length = np.linspace(adelie_flipper_length_obs.min(), adelie_flipper_length_obs.max(), 100)

flipper_length_mean = alpha_m + beta_m * flipper_length
ax.plot(flipper_length, flipper_length_mean, c='C4',
         label=f'y = {alpha_m:.2f} + {beta_m:.2f} * x')

ax.scatter(adelie_flipper_length_obs, adelie_mass_obs)

az.plot_hdi(adelie_flipper_length_obs, idata_adelie_flipper_regression.posterior['μ'], hdi_prob=0.94, color='k', ax=ax)

ax.set_xlabel('Flipper Length')
ax.set_ylabel('Mass');
plt.savefig('img/chp03/flipper_length_mass_regression.png', dpi=300)



### 预测

前面我们估计出了鳍肢长度与体重之间的线性关系。回归的另一个用途,就是利用这种关系来做预测:
给定一只企鹅的鳍肢长度,能否预测它的体重?当然可以。我们会用 `model_adelie_flipper_regression`
的结果来做这件事。因为在贝叶斯统计中我们处理的是分布,所以最终得到的不是单一的预测值,而是
一整个可能取值的分布——也就是前面定义过的后验预测分布。实践中,我们通常不会用解析方法计算
预测,而是用 PPL、基于后验样本来估计。比如,如果有一只鳍肢长度恰好是平均值的企鹅,想用 PyMC
预测它可能的体重,可以这样写:


In [ ]:
# 原书做法是用 pm.set_data 把鳍肢长度改成单个新值,再调用
# pm.sample_posterior_predictive 让 PyMC 重新编译计算图、在新数据点上生成预测。
# 但这个"改变 pm.Data 形状后自动重新编译"的机制依赖 C 编译器后端;在本沙盒环境中
# 我们只有纯 Python/NUMBA 后端(缺少 C 编译器),NUMBA 的 JIT 编译要求形状在编译期
# 静态已知,无法安全地处理这种"运行时改变已观测变量长度"的场景,会报形状不兼容错误。
# 因此这里改用等价的手工计算:直接从后验样本里取出 β_0、β_1、σ,在目标鳍肢长度处
# 计算 μ,再从 Normal(μ, σ) 里抽样,得到的 mass/μ 分布与原书方法在统计上完全等价。
rng = np.random.default_rng(0)
β_0_post = idata_adelie_flipper_regression.posterior["β_0"].values.reshape(-1)
β_1_post = idata_adelie_flipper_regression.posterior["β_1"].values.reshape(-1)
σ_post = idata_adelie_flipper_regression.posterior["σ"].values.reshape(-1)

mu_at_mean_flipper = β_0_post + β_1_post * adelie_flipper_length_obs.mean()
mass_at_mean_flipper = rng.normal(mu_at_mean_flipper, σ_post)

posterior_predictions = az.from_dict(posterior_predictive={
    "mass": mass_at_mean_flipper[None, :],
    "μ": mu_at_mean_flipper[None, :],
})



第一行代码把鳍肢长度固定为观测到的平均鳍肢长度,然后用回归模型
`model_adelie_flipper_regression`,在这个固定值处生成体重的后验预测样本。下图画出了"平均
鳍肢长度企鹅"体重的后验预测分布,以及均值本身的后验。


In [ ]:
fig, ax = plt.subplots()
az.plot_dist(posterior_predictions.posterior_predictive["mass"],
             label="Posterior Predictive of \nIndividual Penguin Mass", ax=ax)
az.plot_dist(posterior_predictions.posterior_predictive["μ"], label="Posterior Predictive of μ", color="C4", ax=ax)
ax.set_xlim(2900, 4500);
ax.legend(loc=2)
ax.set_xlabel("Mass (grams)")
ax.set_yticks([])
plt.savefig('img/chp03/flipper_length_mass_posterior_predictive.png', dpi=300)



简而言之,我们不仅能用前面的模型估计鳍肢长度和体重的关系,还能得到任意给定鳍肢长度下的体重
估计——换句话说,可以用估计出的 $\beta_0$、$\beta_1$ 系数,借助后验预测分布,为任意鳍肢长度
的"未见过的企鹅"做出体重预测。

因此,后验预测分布在贝叶斯语境下是一个特别强大的工具:它不仅能给出最可能的取值,还能给出一整
个纳入了估计不确定性的合理取值分布。

### 中心化(Centering)

前面的模型很好地估计出了鳍肢长度与企鹅体重之间的相关性,也能在给定鳍肢长度时预测体重。可惜
的是,按照当前的数据和模型,我们对 $\beta_0$ 的估计并不是特别有用。不过我们可以通过变换让
$\beta_0$ 变得更好解释——这里我们选用**中心化(centering)**变换,把一组数值的均值平移到零:


In [ ]:
adelie_flipper_length_obs = penguins.loc[adelie_mask, "flipper_length_mm"].values
adelie_flipper_length_c = adelie_flipper_length_obs - adelie_flipper_length_obs.mean()



为便于对照,我们先用 PyMC 拟合中心化后的模型:


In [ ]:
with pm.Model() as model_adelie_flipper_regression:
    σ = pm.HalfStudentT("σ", 100, 2000)
    β_1 = pm.Normal("β_1", 0, 4000)
    β_0 = pm.Normal("β_0", 0, 4000)
    μ = pm.Deterministic("μ", β_0 + β_1*adelie_flipper_length_c)

    mass = pm.Normal("mass", mu=μ, sigma=σ, observed = adelie_mass_obs)

    inf_data_adelie_flipper_length_c = pm.sample(random_seed=0)


In [ ]:
az.plot_posterior(inf_data_adelie_flipper_length_c, var_names = ["β_0", "β_1"], figsize=(10, 4));
plt.savefig("img/chp03/singlespecies_multipleregression_centered.png")



现在用中心化后的协变量,再用 TFP 拟合一遍这个模型。


In [ ]:
adelie_flipper_length_c = adelie_flipper_length_obs - adelie_flipper_length_obs.mean()


In [ ]:
def gen_adelie_flipper_model(adelie_flipper_length):
    adelie_flipper_length = tf.constant(adelie_flipper_length, tf.float32)

    @tfd.JointDistributionCoroutine
    def jd_adelie_flipper_regression():
        σ = yield root(tfd.HalfStudentT(df=100, loc=0, scale=2000, name='sigma'))
        β_1 = yield root(tfd.Normal(loc=0, scale=4000, name='beta_1'))
        β_0 = yield root(tfd.Normal(loc=0, scale=4000, name='beta_0'))
        μ = β_0[..., None] + β_1[..., None] * adelie_flipper_length
        mass = yield tfd.Independent(
            tfd.Normal(loc=μ, scale=σ[..., None]),
            reinterpreted_batch_ndims=1,
            name='mass')

    return jd_adelie_flipper_regression


# 用中心化后的协变量拟合(与上面 PyMC 的中心化模型等价)
jd_adelie_flipper_regression = gen_adelie_flipper_model(
    adelie_flipper_length_c)

mcmc_samples, sampler_stats = run_mcmc(
    1000, jd_adelie_flipper_regression, n_chains=4, num_adaptation_steps=1000,
    mass=tf.constant(adelie_mass_obs, tf.float32))

inf_data_adelie_flipper_length_c = az.from_dict(
    posterior={
        k:np.swapaxes(v, 1, 0)
        for k, v in mcmc_samples._asdict().items()},
    sample_stats={
        k:np.swapaxes(sampler_stats[k], 1, 0)
        for k in ["target_log_prob", "diverging", "accept_ratio", "n_steps"]
    }
)


In [ ]:
az.plot_posterior(inf_data_adelie_flipper_length_c, var_names = ["beta_0", "beta_1"]);



这里用 TFP 定义的数学模型,和前面 PyMC 版本的 `model_adelie_flipper_regression` 完全等价,
唯一区别是对协变量做了中心化处理。从 PPL 的角度看,TFP 的结构要求我们在好几处加上
`tensor_x[..., None]`,把标量批次扩展一维,以便能和向量批次做广播。具体来说,`None`
会追加一个新的轴,等价的写法还有 `np.newaxis` 或 `tf.newaxis`。我们还把模型包在一个函数
里,方便以后用不同的协变量做条件化——这里我们用的是中心化后的鳍肢长度,但换成非中心化的
协变量,结果也会与之前的模型接近。

再画一遍系数图会发现,$\beta_1$ 和 PyMC 模型给出的结果一样,但 $\beta_0$ 的分布变了。由于
我们把输入数据在其均值处做了中心化,现在 $\beta_0$ 的分布,正好等价于之前非中心化数据集下对
"分组均值"的预测。通过中心化,我们现在可以直接把 $\beta_0$ 解读为"具有平均鳍肢长度的 Adelie
企鹅"体重均值的分布。这种对输入变量做变换的思路,也可以在任意选定的取值处进行——比如我们
也可以减去观测到的最小鳍肢长度来拟合模型,这样 $\beta_0$ 的含义就会变成"鳍肢长度最小的那些
企鹅"的均值分布。关于线性回归中变换的更多讨论,推荐参阅《Applied Regression Analysis and
Generalized Linear Models》一书。

## 多元线性回归

许多物种都存在性二态性(sexual dimorphism,即雌雄个体存在系统性差异)。事实上,对企鹅性
二态性的研究,正是最初收集 Palmer Penguins 数据集的动机之一。为了更深入地研究企鹅的性二态性,
我们再加入第二个协变量——性别,把它编码成类别变量,看看能否更精确地估计企鹅体重。


In [ ]:
# 把类别型预测变量二元编码
# 注意这里补充了 .astype(int).values:较新版本 pandas 会把字符串列读成
# 扩展的 StringDtype,对它调用 .replace({...}) 把值换成 0/1 之后,
# 结果并不会像旧版 pandas 那样自动降级为数值 dtype,而是停留在
# dtype=object 的数组上。这样的 object 数组和 pytensor 张量做运算时,
# 会被 numpy 自己的逐元素运算符接管,得到一个 dtype=object 的数组而不是
# 真正的 pytensor 张量表达式,传给 pm.Deterministic 时会报错,因此这里
# 显式转换成 int
sex_obs = penguins.loc[adelie_mask ,"sex"].replace({"male":0, "female":1}).astype(int).values

with pm.Model() as model_penguin_mass_categorical:
    σ = pm.HalfStudentT("σ", 100, 2000)
    β_0 = pm.Normal("β_0", 0, 3000)
    β_1 = pm.Normal("β_1", 0, 3000)
    β_2 = pm.Normal("β_2", 0, 3000)

    μ = pm.Deterministic(
        "μ", β_0 + β_1 * adelie_flipper_length_obs + β_2 * sex_obs)

    mass = pm.Normal("mass", mu=μ, sigma=σ, observed=adelie_mass_obs)

    inf_data_penguin_mass_categorical = pm.sample(target_accept=.9)



你会注意到多了一个新参数 $\beta_2$,参与到 $\mu$ 的取值中。由于性别是类别型预测变量(本例中
只有雌雄两类),我们分别编码为 1 和 0。对模型来说,这意味着雌性企鹅的 $\mu$ 是三项之和,而
雄性企鹅的 $\mu$ 是两项之和(因为 $\beta_2$ 那一项会被清零)。


In [ ]:
az.plot_posterior(inf_data_penguin_mass_categorical, var_names =["β_0", "β_1", "β_2"], figsize=(10, 4))
plt.savefig("img/chp03/adelie_sex_coefficient_posterior.png")


In [ ]:
az.summary(inf_data_penguin_mass_categorical, var_names=["β_0","β_1","β_2", "σ"])



> **线性模型的语法糖**:线性模型的应用极其广泛,以至于专门有人为回归写了特殊的语法、方法和
> 库。Bambi(BAyesian Model-Building Interface)就是其中之一——一个用公式语法拟合广义线性
> 层级模型的 Python 包,风格类似 R 语言里的 lme4、nlme、rstanarm 或 brms 等包。Bambi 底层
> 使用 PyMC,提供了更高层的 API。如果不考虑先验的具体设置,上面那个模型用 Bambi 大致可以
> 写成:
>
> ```python
> import bambi as bmb
> model = bmb.Model("body_mass_g ~ flipper_length_mm + sex", penguins[adelie_mask])
> trace = model.fit()
> ```
>
> 如果没有指定先验,Bambi 会自动分配。在内部,Bambi 几乎保存了 PyMC 生成的所有对象,方便
> 用户获取、检查和修改。此外 Bambi 还会返回一个 `az.InferenceData` 对象,可以直接配合
> ArviZ 使用。

由于我们把男性(male)编码为 0,`model_penguin_mass_categorical` 给出的后验,估计的是"在
鳍肢长度相同的前提下",雌性 Adelie 企鹅与雄性相比,体重的差异。这一点很重要:一旦加入了第二
个协变量,我们就有了一个多元线性回归,解读系数时需要更加谨慎——此时系数表示的是:**在其他
协变量保持不变的前提下**,某个协变量与响应变量之间的关系。


In [ ]:
axes = az.plot_forest(
    [idata_adelie_mass, idata_adelie_flipper_regression, inf_data_penguin_mass_categorical],
    model_names=["mass_only", "flipper_regression", "flipper_sex_regression"],
    var_names=["σ"], combined=True, figsize=(10, 2))

axes[0].set_title("σ Comparison 94.0 HDI")
plt.savefig("img/chp03/singlespecies_multipleregression_forest_sigma_comparison.png")


In [ ]:
# 修正配色
fig, ax = plt.subplots(figsize=(10, 4))
alpha_1 = inf_data_penguin_mass_categorical.posterior["β_0"].mean().item()
beta_1 = inf_data_penguin_mass_categorical.posterior["β_1"].mean().item()
beta_2 = inf_data_penguin_mass_categorical.posterior["β_2"].mean().item()


flipper_length = np.linspace(adelie_flipper_length_obs.min(), adelie_flipper_length_obs.max(), 100)

mass_mean_male = alpha_1 + beta_1 * flipper_length
mass_mean_female = alpha_1 + beta_1 * flipper_length + beta_2

ax.plot(flipper_length, mass_mean_male,
         label="Male")

ax.plot(flipper_length, mass_mean_female, c='C4',
         label="Female")

ax.scatter(adelie_flipper_length_obs, adelie_mass_obs, c=[{0:"k", 1:"b"}[code] for code in sex_obs])

ax.set_xlabel('Flipper Length')
ax.set_ylabel('Mass');
ax.legend()
plt.savefig("img/chp03/single_species_categorical_regression.png")



我们再一次比较三个模型的标准差,看看是否进一步降低了估计的不确定性——的确如此,额外信息再次
帮助改善了估计。这次 $\sigma$ 的均值,从"无协变量模型"里的 462 克,降到了"包含鳍肢长度与
性别两个协变量的线性模型"里的 298 克。这种不确定性的下降说明,性别确实为估计企鹅体重提供了
有用信息。

> **协变量并非越多越好**:任何模型拟合算法都会去寻找信号——哪怕这个"信号"其实只是随机噪声。
> 这种现象被称为过拟合:算法在已见过的样本上能很好地把协变量映射到结果,却无法泛化到新的
> 观测。在线性回归中,我们可以通过生成 100 个随机协变量、拟合到一个随机模拟数据集上来演示
> 这一点——即便协变量与目标毫无关系,我们也很可能被"线性模型表现不错"这个假象误导。


In [ ]:
def gen_jd_flipper_bill_sex(flipper_length, sex, bill_length, dtype=tf.float32):
    flipper_length, sex, bill_length = tf.nest.map_structure(
        lambda x: tf.constant(x, dtype),
        (flipper_length, sex, bill_length)
    )

    @tfd.JointDistributionCoroutine
    def jd_flipper_bill_sex():
        σ = yield root(
            tfd.HalfStudentT(df=100, loc=0, scale=2000, name="sigma"))
        β_0 = yield root(tfd.Normal(loc=0, scale=3000, name="beta_0"))
        β_1 = yield root(tfd.Normal(loc=0, scale=3000, name="beta_1"))
        β_2 = yield root(tfd.Normal(loc=0, scale=3000, name="beta_2"))
        β_3 = yield root(tfd.Normal(loc=0, scale=3000, name="beta_3"))
        μ = (β_0[..., None]
             + β_1[..., None] * flipper_length
             + β_2[..., None] * sex
             + β_3[..., None] * bill_length
            )
        mass = yield tfd.Independent(
            tfd.Normal(loc=μ, scale=σ[..., None]),
            reinterpreted_batch_ndims=1,
            name="mass")

    return jd_flipper_bill_sex

bill_length_obs = penguins.loc[adelie_mask, "bill_length_mm"]
jd_flipper_bill_sex = gen_jd_flipper_bill_sex(
    adelie_flipper_length_obs, sex_obs, bill_length_obs)

mcmc_samples, sampler_stats = run_mcmc(
    1000, jd_flipper_bill_sex, n_chains=4, num_adaptation_steps=1000,
    mass=tf.constant(adelie_mass_obs, tf.float32))


In [ ]:
idata_model_penguin_flipper_bill_sex = az.from_dict(
    posterior={
        k:np.swapaxes(v, 1, 0)
        for k, v in mcmc_samples._asdict().items()},
    sample_stats={
        k:np.swapaxes(sampler_stats[k], 1, 0)
        for k in ["target_log_prob", "diverging", "accept_ratio", "n_steps"]}
)


In [ ]:
az.plot_posterior(idata_model_penguin_flipper_bill_sex, var_names=["beta_1", "beta_2", "beta_3"]);


In [ ]:
az.summary(idata_model_penguin_flipper_bill_sex, var_names=["beta_1", "beta_2", "beta_3", "sigma"])



### 反事实分析(Counterfactuals)

前面我们用单协变量模型拟合出的参数做过预测,通过改变鳍肢长度这一个协变量,得到了固定鳍肢长度
下的体重估计。在多元回归里,我们也可以做类似的事:固定除某一个协变量之外的所有协变量,观察
这一个协变量的变化会如何改变预期结果——这种分析叫做**反事实分析(counterfactual
analysis)**。下面我们扩展上一节的多元回归,加入喙长(bill length),并在 TFP 中做一次反事实
分析。上面已经给出了模型搭建与推断的代码,新增了系数 `beta_3` 对应新加入的喙长协变量。推断
完成后,我们可以模拟"虚构鳍肢长度"下企鹅的体重——固定性别为雄性,喙长固定为数据集的观测均值。
由于我们把模型生成过程包在了一个 Python 函数里(函数式编程风格),可以很方便地用新的预测变量
做条件化,这对反事实分析很有用。


In [ ]:
mean_flipper_length = penguins.loc[adelie_mask, "flipper_length_mm"].mean()
# 反事实维度设为 21,方便刚好取到均值这个点
counterfactual_flipper_lengths = np.linspace(
    mean_flipper_length-20, mean_flipper_length+20, 21)
sex_male_indicator = np.zeros_like(counterfactual_flipper_lengths)
mean_bill_length = np.ones_like(
    counterfactual_flipper_lengths) * bill_length_obs.mean()

jd_flipper_bill_sex_counterfactual = gen_jd_flipper_bill_sex(
    counterfactual_flipper_lengths, sex_male_indicator, mean_bill_length)
ppc_samples = jd_flipper_bill_sex_counterfactual.sample(value=mcmc_samples)
estimated_mass = ppc_samples[-1].numpy().reshape(-1, 21)


In [ ]:
_, ax = plt.subplots(figsize=(10, 3))
az.plot_hdi(counterfactual_flipper_lengths, estimated_mass, color="C2", plot_kwargs={"ls": "--"}, ax=ax)
ax.plot(counterfactual_flipper_lengths, estimated_mass.mean(axis=0), lw=4, c="blue")
ax.set_title("Mass estimates with Flipper Length Counterfactual for \n Male Penguins at Mean Observed Bill Length")
ax.set_xlabel("Counterfactual Flipper Length")
ax.set_ylabel("Mass")
plt.savefig("img/chp03/linear_counter_factual.png")



上图是所谓的反事实图(counterfactual plot)。顾名思义,"反事实"意味着我们在评估一种与
已观测数据(即"事实")相反的情形——换句话说,我们在评估那些从未发生过的情形。反事实图最简单
的用法,就是像我们刚才做的那样,调整某个协变量,观察结果——这让我们能够探索原本难以触及的
"假如……会怎样"场景 [^6]。但我们在解读这类"戏法"时必须保持谨慎。第一个陷阱是:反事实取值
可能根本不可能存在——比如现实中或许根本不存在鳍肢长度超过 1500 毫米的企鹅,但模型依然会
欣然给出这只"虚构企鹅"的估计。第二个陷阱更隐蔽:我们假设可以独立地改变每个协变量,但现实中
未必如此——比如企鹅鳍肢变长时,喙长可能也会随之变化。反事实分析的强大之处在于,它能让我们
探索那些没有发生过、或者至少我们没有观测到的结果。但它也很容易让我们对那些*永远不会*发生的
情形生成估计——模型本身分不清这两者的区别,分辨这一点是建模者的责任。

> **相关性与因果性**:在解读线性回归时,很容易脱口而出"$X$ 增加**导致**了 $Y$ 增加"。但这
> 未必成立——事实上,仅凭一个(线性)回归本身,是无法做出因果论断的。数学上,线性模型把两个
> (或更多)变量关联在一起,但这种关联未必是因果的。比如,给植物浇更多水确实(而且是因果性地)
> 会促进植物生长(至少在一定范围内),但没有什么能阻止我们在模型中把这个关系反过来,用植物
> 的生长情况去估计降雨量——即便植物生长本身并不会导致降雨 [^7]。统计学中的因果推断
> (Causal Inference)这一分支,正是研究如何在随机实验或观察性研究的语境下,得出因果论断
> 所需的工具和流程(第 7 章会简要讨论)。

## 广义线性模型

目前为止讨论的所有线性模型,都假设观测值在给定条件下服从高斯分布,这在很多场景下都行得通。但
有时我们想用其他分布——比如建模被限制在某个区间内的量,像 $[0, 1]$ 区间内的概率,或者自然数
$\{1, 2, 3, \dots \}$ 这样的计数事件。要做到这一点,我们把线性函数 $\mathbf{X} \mathit{\beta}$
用一个反连接函数(inverse link function)[^8] $\phi$ 做变换:

$$
\begin{split}
\mu =& \phi(\mathbf{X} \beta) \\
Y \sim& \Psi (\mu, \theta)
\end{split}
$$

其中 $\Psi$ 是由 $\mu$、$\theta$ 参数化的某个分布,代表数据的似然。

反连接函数的具体作用,是把取值范围在整个实数轴 $(-\infty, \infty)$ 上的输出,映射到参数所
限定的那个受限区间。换句话说,反连接函数正是把线性模型推广到更多模型架构所需要的那个"戏法"。
从这个意义上说,我们处理的仍然是线性模型——因为生成观测值的分布的期望,依然是参数和协变量的
线性函数——但这个"戏法"让我们能把线性模型的应用场景大大拓展 [^9]。

### 逻辑回归

最常见的广义线性模型之一就是逻辑回归(logistic regression)。它特别适合为"只有两种可能结果"
的数据建模——我们观测到的要么是这个,要么是那个。抛硬币正反面的概率是最典型的教科书例子。更
"接地气"的例子包括:制造业里出现缺陷的概率、癌症检测呈阳性或阴性、火箭发射失败的概率等等。
在逻辑回归中,反连接函数不出所料地被称为逻辑函数(logistic function),它把
$(-\infty, \infty)$ 映射到 $(0,1)$ 区间。这非常方便,因为这样我们就能把线性函数映射到"概率
参数"所要求的取值范围——按照定义,概率必须落在 0 到 1 之间。

$$
p = \frac{1}{1+e^{-\mathbf{X}\beta}}
$$

有了逻辑回归,我们就能用线性模型来估计某个事件的概率。有时候,我们想做的不是估计概率,而是
**分类**——即给定数据,预测一个具体的类别。为此我们需要把 $(-\infty, \infty)$ 区间上的连续
预测,转换成 0 到 1 之间的取值,再借助一个决策边界,把预测归到 $\{0,1\}$ 这个集合里。假设我们
把决策边界设在概率 0.5 处。对一个只有截距和一个协变量的模型,有:

$$
\begin{split}
0.5 &= logistic(\beta_{0} + \beta_{1}*x) \\
logit(0.5) &= \beta_{0} + \beta_{1}*x \\
0 &= \beta_{0} + \beta_{1}*x \\
x &= -\frac{\beta_{0}}{\beta_{1}} \\
\end{split}
$$

注意 $logit$ 是 $logistic$ 的反函数。也就是说,一旦拟合出逻辑模型,我们就可以用系数
$\beta_0$、$\beta_1$,直接算出使得类别概率大于 0.5 的 $x$ 取值。

### 给企鹅分类

前面几节,我们用企鹅的性别和喙长来估计体重。现在换个问法:如果给定一只企鹅的体重、性别和喙长,
能否预测它的物种?我们用 Adelie 和 Chinstrap 两个物种,把问题变成一个二分类任务。像之前一样,
我们先从一个只有一个协变量(喙长)的简单模型入手:


In [ ]:
species_filter = penguins["species"].isin(["Adelie", "Chinstrap"])
bill_length_obs = penguins.loc[species_filter, "bill_length_mm"].values
species = pd.Categorical(penguins.loc[species_filter, "species"])

with pm.Model() as model_logistic_penguins_bill_length:
    β_0 = pm.Normal("β_0", mu=0, sigma=10)
    β_1 = pm.Normal("β_1", mu=0, sigma=10)

    μ = β_0 + pm.math.dot(bill_length_obs, β_1)

    # 应用 sigmoid 连接函数
    θ = pm.Deterministic("θ", pm.math.sigmoid(μ))

    # 便于之后画决策边界
    bd = pm.Deterministic("bd", -β_0/β_1)

    # 注意这里似然换成了伯努利分布
    yl = pm.Bernoulli("yl", p=θ, observed=species.codes)

    idata_logistic_penguins_bill_length = pm.sample(5000, chains=2, random_seed=0,
                                                    idata_kwargs={"log_likelihood":True})
    idata_logistic_penguins_bill_length.extend(pm.sample_prior_predictive(samples=10000))
    idata_logistic_penguins_bill_length.extend(pm.sample_posterior_predictive(idata_logistic_penguins_bill_length))



在广义线性模型中,"参数先验 → 响应变量"这条映射链有时会更难直观理解。我们可以借助先验预测
样本,帮助我们可视化预期的观测结果。在这个企鹅分类的例子里,在看到任何数据之前,我们认为
"不论喙长如何,Chinstrap 企鹅和 Adelie 企鹅出现的概率应该差不多"是合理的假设。我们可以用
先验预测分布,double-check 一下这个建模意图是否被先验和模型正确地表达了出来。下图显示,在
看到数据之前,两个类别大致均衡——这正是我们期望看到的结果。


In [ ]:
_, ax = plt.subplots(figsize=(10, 2))
az.plot_dist(idata_logistic_penguins_bill_length.prior_predictive["yl"], color="C2", ax=ax)
ax.set_xticklabels(["Adelie: 0", "Chinstrap: 1"] )
plt.savefig("img/chp03/prior_predictive_logistic.png")


In [ ]:
az.plot_trace(idata_logistic_penguins_bill_length, var_names=["β_0", "β_1"], kind="rank_bars");



拟合出参数之后,我们可以用 `az.summary(.)` 检查系数。虽然能读出系数的数值,但它们不像线性
回归里那样能直接解读。从 $\beta_1$ 为正、且其 HDI 不跨越零可以看出,喙长与物种之间确实存在
某种关系。决策边界也可以比较直接地解读:大约 44 毫米的喙长,是两个物种之间大致的分界点。把
回归结果画出来(下图)会更直观:我们能看到熟悉的逻辑曲线,从左边的 0 变化到右边的 1,决策
边界的位置也和数据本身的分布相符。


In [ ]:
az.summary(idata_logistic_penguins_bill_length, var_names=["β_0", "β_1"], kind="stats")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

theta = idata_logistic_penguins_bill_length.posterior["θ"].mean(("chain", "draw"))


idx = np.argsort(bill_length_obs)

# 决策边界
ax.vlines(idata_logistic_penguins_bill_length.posterior["bd"].values.mean(), 0, 1, color='k')
bd_hpd = az.hdi(idata_logistic_penguins_bill_length.posterior["bd"].values.flatten(), ax=ax)
plt.fill_betweenx([0, 1], bd_hpd[0], bd_hpd[1], color='C2', alpha=0.5)


for i, (label, marker) in enumerate(zip(species.categories, (".", "s"))):
    _filter = (species.codes == i)
    x = bill_length_obs[_filter]
    y = np.random.normal(i, 0.02, size=_filter.sum())
    ax.scatter(bill_length_obs[_filter], y, marker=marker, label=label, alpha=.8)

az.plot_hdi(bill_length_obs, idata_logistic_penguins_bill_length.posterior["θ"].values, color='C4', ax=ax, plot_kwargs={"zorder":10})
ax.plot(bill_length_obs[idx], theta[idx], color='C4', zorder=10)

ax.set_xlabel("Bill Length (mm)")
ax.set_ylabel('θ', rotation=0)
plt.legend()
plt.savefig("img/chp03/logistic_bill_length.png")



我们再换个思路,依然是分类企鹅,这次改用体重作为协变量:


In [ ]:
mass_obs = penguins.loc[species_filter, "body_mass_g"].values

with pm.Model() as model_logistic_penguins_mass:
    β_0 = pm.Normal("β_0", mu=0, sigma=10)
    β_1 = pm.Normal("β_1", mu=0, sigma=10)

    μ = β_0 + pm.math.dot(mass_obs, β_1)
    θ = pm.Deterministic("θ", pm.math.sigmoid(μ))
    bd = pm.Deterministic("bd", -β_0/β_1)

    yl = pm.Bernoulli("yl", p=θ, observed=species.codes)


    idata_logistic_penguins_mass = pm.sample(5000, chains=2,
                                             target_accept=.9, random_seed=0,
                                             idata_kwargs={"log_likelihood":True})
    idata_logistic_penguins_mass.extend(pm.sample_posterior_predictive(idata_logistic_penguins_mass))


In [ ]:
az.plot_trace(idata_logistic_penguins_mass, var_names=["β_0", "β_1"], kind="rank_bars");



数值汇总显示 $\beta_1$ 的估计值接近 0,说明单凭体重这个协变量,并没有足够的信息来区分这两个
类别。这不一定是坏事,只是模型在告诉我们:这两个物种的体重差异,在它看来并不明显。把数据和
逻辑回归拟合结果画出来之后,这一点会更加明显。


In [ ]:
az.summary(idata_logistic_penguins_mass, var_names=["β_0", "β_1", "bd"], kind="stats")


In [ ]:
species.codes


In [ ]:
theta = idata_logistic_penguins_mass.posterior['θ'].mean(("chain", "draw"))
bd = idata_logistic_penguins_mass.posterior['bd']

fig, ax = plt.subplots()
idx = np.argsort(mass_obs)

ax.plot(mass_obs[idx], theta[idx], color='C4', lw=3)
for i, (label, marker) in enumerate(zip(species.categories, (".", "s"))):
    _filter = (species.codes == i)
    x = mass_obs[_filter]
    y = np.random.normal(i, 0.02, size=_filter.sum())
    ax.scatter(mass_obs[_filter], y, marker=marker, label=label, alpha=.8)



az.plot_hdi(mass_obs, idata_logistic_penguins_mass.posterior['θ'], color='C4', ax=ax)

ax.set_xlabel("Mass (Grams)")
ax.set_ylabel('θ', rotation=0)
plt.legend()

plt.savefig("img/chp03/logistic_mass.png")



不应该因为这次没找到关系就气馁——有效的建模本来就包含反复试错。这不是说要随便乱试、指望
碰运气,而是说可以放心地借助计算工具,为下一步该怎么走提供线索。

下面我们同时用喙长和体重两个协变量,做一次多元逻辑回归,再画一遍决策边界。这次图的坐标轴略
有不同——纵轴不再是类别概率,而是体重,这样我们能看到两个自变量之间的决策边界。这些可视化
检查很有帮助,但终究是主观的,我们也可以用诊断量来给拟合结果打分。


In [ ]:
X = penguins.loc[species_filter, ["bill_length_mm", "body_mass_g"]]

# 加一列全 1 表示截距
X.insert(0,"Intercept", value=1)
X = X.values

with pm.Model() as model_logistic_penguins_bill_length_mass:
    β = pm.Normal("β", mu=0, sigma=20, shape=3)

    μ = pm.math.dot(X, β)

    θ = pm.Deterministic("θ", pm.math.sigmoid(μ))
    bd = pm.Deterministic("bd", -β[0]/β[2] - β[1]/β[2] * X[:,1])

    yl = pm.Bernoulli("yl", p=θ, observed=species.codes)

    idata_logistic_penguins_bill_length_mass = pm.sample(5000, chains=2,
                                                         random_seed=0,
                                                         target_accept=.9,
                                                         idata_kwargs={"log_likelihood":True})
    idata_logistic_penguins_bill_length_mass.extend(pm.sample_posterior_predictive(idata_logistic_penguins_bill_length_mass))


In [ ]:
az.plot_trace(idata_logistic_penguins_bill_length_mass, compact=False, var_names=["β"], kind="rank_bars");


In [ ]:
az.summary(idata_logistic_penguins_bill_length_mass, var_names=["β"])


In [ ]:
fig,ax  = plt.subplots()
idx = np.argsort(X[:,1])
bd = idata_logistic_penguins_bill_length_mass.posterior["bd"].mean(("chain", "draw"))[idx]


species_filter = species.codes.astype(bool)

# 线性拟合
ax.plot(X[:,1][idx], bd, color='C4');
az.plot_hdi(X[:,1], idata_logistic_penguins_bill_length_mass.posterior["bd"], color='C4', ax=ax)

# 散点
ax.scatter(X[~species_filter,1], X[~species_filter,2], alpha=.8,  label="Adelie", zorder=10)
ax.scatter(X[species_filter,1], X[species_filter,2], marker="s", label="Chinstrap", zorder=10)


ax.set_ylabel("Mass (grams)")
ax.set_xlabel("Bill Length (mm)")


ax.legend()
plt.savefig("img/chp03/decision_boundary_logistic_mass_bill_length.png");



要评估逻辑回归模型的拟合效果,可以用分离图(separation plot)。分离图是一种评估二元观测数据
模型校准情况的方法:它按类别对预测值排序展示,理想情况下——如果模型能完美区分两类——图上会
呈现两个界限分明的矩形块。我们的例子里,没有一个模型完美区分了两个物种,但包含喙长的模型,
明显比只有体重的模型表现更好。一般来说,贝叶斯分析的目标并不是"完美校准",但分离图(以及像
LOO-PIT 这样的其他校准评估方法)依然能帮助我们比较模型、发现可以改进的地方。


In [ ]:
models = {"bill": idata_logistic_penguins_bill_length,
          "mass": idata_logistic_penguins_mass,
          "mass bill": idata_logistic_penguins_bill_length_mass}

_, axes = plt.subplots(3, 1, figsize=(12, 4), sharey=True)
for (label, model), ax in zip(models.items(), axes):
    az.plot_separation(model, "yl", ax=ax, color="C4")
    ax.set_title(label)

plt.savefig("img/chp03/penguins_separation_plot.png")



我们也可以用 LOO 来比较刚才建立的三个模型——只用体重、只用喙长,以及同时使用两者。按照 LOO,
只用体重的模型在区分物种上表现最差,只用喙长的模型居中,同时使用喙长和体重的模型表现最好。
这和我们从图中看到的结果一致,现在又有了数值上的印证。


In [ ]:
az.compare({"mass": idata_logistic_penguins_mass,
            "bill": idata_logistic_penguins_bill_length,
            "mass_bill": idata_logistic_penguins_bill_length_mass}).round(1)



### 解读对数几率(Log Odds)

逻辑回归中,斜率告诉你的是:$x$ 每增加一个单位,对数几率(log odds)会增加多少个单位。所谓
**几率(odds)**,最简单来说就是"事件发生的概率"与"事件不发生的概率"之比。比如,在企鹅例子
中,如果从 Adelie 和 Chinstrap 企鹅里随机挑一只,挑到 Adelie 的概率是 0.68:


In [ ]:
# 各物种的数量
counts = penguins["species"].value_counts()
adelie_count = counts["Adelie"],
chinstrap_count = counts["Chinstrap"]
adelie_count / (adelie_count + chinstrap_count)



而对同一个事件,几率则是:


In [ ]:
adelie_count / chinstrap_count



几率和概率由相同的成分构成,只是经过了一种变换,让"一个事件相对另一个事件发生的比率"更容易
解读。用几率来表达的话:如果从 Adelie 和 Chinstrap 企鹅中随机抽样,我们预期 Adelie 企鹅的
数量会是 Chinstrap 企鹅的 2.14 倍。

利用几率的概念,我们可以定义 logit:logit 就是几率的自然对数。我们可以用 logit,把逻辑回归
公式改写成另一种形式:

$$
\log \left(\frac{p}{1-p} \right) = \boldsymbol{X} \beta
$$

这种改写方式,让我们能把逻辑回归的系数,解读为"对数几率的变化量"。借助这一点,我们可以计算
"观测到的喙长发生变化"时,Adelie 相对 Chinstrap 企鹅出现概率的变化,如下所示。这类变换不仅
在数学上很有意思,在讨论统计结果时也非常实用,我们会在后续章节更深入地讨论这个话题。


In [ ]:
β_0 = idata_logistic_penguins_bill_length.posterior["β_0"].mean().item()
β_1 = idata_logistic_penguins_bill_length.posterior["β_1"].mean().item()


In [ ]:
β_0


In [ ]:
β_1


In [ ]:
bill_length = 45
val_1 = β_0 + β_1 * bill_length
val_2 = β_0 + β_1 * (bill_length+1)

f"Class Probability change from 45mm Bill Length to 46mm: {(special.expit(val_2) - special.expit(val_1))*100:.0f}%"


In [ ]:
bill_length = np.array([30, 45])
val_1 = β_0 + β_1 * bill_length
val_2 = β_0 + β_1 * (bill_length+1)


In [ ]:
special.expit(val_2) - special.expit(val_1)



## 回归模型中先验的选择

熟悉了广义线性模型之后,我们把重点放到先验及其对后验估计的影响上。这里借用《Regression and
Other Stories》一书中的一个例子:一项研究探讨了"父母的外貌吸引力"与"其子女中女孩比例"之间的
关系。研究者把美国青少年的外貌吸引力按 5 档评分,这些受试者后来大多有了孩子,研究者统计了
每个吸引力档位下的性别比例。数据如下,同一段代码里我们也写了一个单变量回归模型——不过这次,
我们特别关注先验和似然应该**放在一起**评估,而不是各自独立地看待。


In [ ]:
x = np.arange(-2,3,1)
y = [50, 44, 50, 47, 56]


In [ ]:
import matplotlib.ticker as mtick
fig, ax = plt.subplots()

ax.scatter(x, y)
ax.set_xticks(x)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_ylim(40, 60)
ax.set_xlabel("Attractiveness of Parent")
ax.set_ylabel("% of Girl Babies")
ax.set_title("Attractiveness of Parent and Sex Ratio")
plt.savefig("img/chp03/beautyratio.png")



名义上,我们会假设男孩女孩出生比例大致相等,且外貌吸引力对性别比例没有影响。这意味着截距
$\beta_0$ 的先验均值设为 50,系数 $\beta_1$ 的先验均值设为 0。同时我们把先验的离散程度设得
很宽,以表达我们对截距、以及吸引力对性别比例的影响都缺乏了解。这并不是完全"无信息"的先验
(我们在前面章节讨论过那种先验),而是一个非常宽泛的先验。


In [ ]:
with pm.Model() as model_uninformative_prior_sex_ratio:
    σ = pm.Exponential("σ", .5)
    β_1 = pm.Normal("β_1", 0, 20)
    β_0 = pm.Normal("β_0", 50, 20)

    μ = pm.Deterministic("μ", β_0 + β_1 * x)

    ratio = pm.Normal("ratio", mu=μ, sigma=σ, observed=y)

    idata_uninformative_prior_sex_ratio = pm.sample(random_seed=0)
    idata_uninformative_prior_sex_ratio.extend(pm.sample_prior_predictive(samples=10000))


In [ ]:
az.plot_posterior(idata_uninformative_prior_sex_ratio.prior, var_names=["β_0", "β_1"])
plt.savefig("img/chp03/priorpredictiveuninformativeKDE.png")


In [ ]:
az.summary(idata_uninformative_prior_sex_ratio, var_names=["β_0", "β_1", "σ"], kind="stats")



用这样的设置拟合模型、运行推断、生成后验样本后,我们估计出 $\beta_1$ 的均值约为 1.4,也就是
说吸引力最低的组和最高的组相比,出生性别比平均会相差 7.4%。如果把不确定性也考虑进去,从条件
化数据之前的 50 条"可能拟合线"随机样本来看,这个比例每单位吸引力甚至可以变化超过 20%
[^10]。


In [ ]:
fig, axes = plt.subplots(2,1, figsize=(5.5, 6), sharex=True)

np.random.seed(0)
# 从先验中取 50 个样本
num_samples = 50
subset = az.extract(idata_uninformative_prior_sex_ratio, group="prior", num_samples=50)

# 画线
axes[0].plot(x, (subset["β_0"]+subset["β_1"]*xr.DataArray(x)).T, c="black", alpha=.3)

# 加上中位数线
b_0_hat = idata_uninformative_prior_sex_ratio.prior["β_0"].values.mean()
b_1_hat = idata_uninformative_prior_sex_ratio.prior["β_1"].values.mean()

axes[0].plot(x, b_0_hat+b_1_hat*x, c="C4", linewidth=4)


# 加散点
axes[0].scatter(x, y)
axes[0].set_xticks(x)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
axes[0].set_ylim(40, 60)
axes[0].set_ylabel("% of Girl Babies")
axes[0].set_title("Prior samples from informative priors");


np.random.seed(0)
num_samples = 50
subset = az.extract(idata_uninformative_prior_sex_ratio, group="prior", num_samples=50)

axes[1].plot(x, (subset["β_0"]+subset["β_1"]*xr.DataArray(x)).T, c="black", alpha=.3)

b_0_hat = idata_uninformative_prior_sex_ratio.posterior["β_0"].values.mean()
b_1_hat = idata_uninformative_prior_sex_ratio.posterior["β_1"].values.mean()

axes[1].plot(x, b_0_hat+b_1_hat*x, c="C4", linewidth=4)

axes[1].scatter(x, y)
axes[1].set_xticks(x)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
axes[1].set_ylim(40, 60)

axes[1].set_xlabel("Attractiveness of Parent")
axes[1].set_ylabel("% of Girl Babies")
axes[1].set_title("Posterior samples from informative priors")



plt.savefig("img/chp03/posterioruninformativelinearregression.png")



从数学角度看,这个结果是"成立"的。但从我们的常识、以及本研究之外对出生性别比的一般了解来看,
这个结果相当可疑。已有研究测得,自然出生性别比大约是每 105 个男孩对应 100 个女孩(范围大致在
103 到 107 之间),换算下来女孩比例约为 48.5%,标准差约为 0.5。而且,即便是那些与人类生物学
关系更密切的因素,对出生性别比的影响也远没有达到这种量级,这就削弱了"外貌吸引力这种主观因素
能有这么大影响"的说法。基于这些信息,两组之间出现 8% 的差异,需要非同寻常的证据支撑才站得
住脚。

我们用更符合这些常识的信息性先验,重新拟合一次模型。画出后验样本后可以看到,系数的集中程度
更高,后验拟合线也落在了考虑现实中合理性别比之后更合理的范围内。


In [ ]:
with pm.Model() as model_informative_prior_sex_ratio:
    σ = pm.Exponential("σ", .5)

    # 注意这里换成了信息量更大的先验
    β_1 = pm.Normal("β_1", 0, .5)
    β_0 = pm.Normal("β_0", 48.5, .5)

    μ = pm.Deterministic("μ", β_0 + β_1 * x)
    ratio = pm.Normal("ratio", mu=μ, sigma=σ, observed=y)

    idata_informative_prior_sex_ratio = pm.sample(random_seed=0)
    idata_informative_prior_sex_ratio.extend(pm.sample_prior_predictive(samples=10000))


In [ ]:
az.summary(idata_informative_prior_sex_ratio, var_names=["β_0", "β_1", "σ"], kind="stats")


In [ ]:
fig, axes = plt.subplots(2,1, figsize=(5.5, 6), sharex=True)

np.random.seed(0)
num_samples = 50
subset = az.extract(idata_informative_prior_sex_ratio, group="prior", num_samples=50)

axes[0].plot(x, (subset["β_0"]+subset["β_1"]*xr.DataArray(x)).T, c="black", alpha=.3)

b_0_hat = idata_informative_prior_sex_ratio.prior["β_0"].values.mean()
b_1_hat = idata_informative_prior_sex_ratio.prior["β_1"].values.mean()

axes[0].plot(x, b_0_hat+b_1_hat*x, c="C4", linewidth=4)


axes[0].scatter(x, y)
axes[0].set_xticks(x)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
axes[0].set_ylim(40, 60)
axes[0].set_ylabel("% of Girl Babies")
axes[0].set_title("Prior samples from informative priors");


np.random.seed(0)
num_samples = 50
subset = az.extract(idata_informative_prior_sex_ratio, group="prior", num_samples=50)

axes[1].plot(x, (subset["β_0"]+subset["β_1"]*xr.DataArray(x)).T, c="black", alpha=.3)

b_0_hat = idata_informative_prior_sex_ratio.posterior["β_0"].values.mean()
b_1_hat = idata_informative_prior_sex_ratio.posterior["β_1"].values.mean()

axes[1].plot(x, b_0_hat+b_1_hat*x, c="C4", linewidth=4)

axes[1].scatter(x, y)
axes[1].set_xticks(x)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
axes[1].set_ylim(40, 60)

axes[1].set_xlabel("Attractiveness of Parent")
axes[1].set_ylabel("% of Girl Babies")
axes[1].set_title("Posterior samples from informative priors")


b_0_hat, b_1_hat
plt.savefig("img/chp03/posteriorinformativelinearregression.png")



这一次我们看到,吸引力对性别的估计影响变得微乎其微——数据中根本没有足够的信息来撼动后验。
正如我们在前面章节所说,先验的选择既是负担也是恩赐。不管你更倾向于哪种看法,重要的是:使用
这个统计工具时,要有一个可解释、有原则依据的选择。

## 习题

本章习题涉及大量代码练习(把 PyMC 模型翻译成 TFP、反之亦然;修改协变量;增删先验信息量;
比较模型等),完整题目请参阅原书英文版（`markdown/chp_03.md` 第 1904 行起,共约 20 道题,
难度从 E(易)到 H(难)分级）。这里概述其涵盖的练习方向,供读者按需练习:

- 比较统计量与分组逻辑的日常类比(E1);企鹅体重模型的 MCSE/ESS/$\hat R$ 诊断练习(E2)。
- 用自己的话解释协变量估计、预测、反事实分析三者的区别与联系(E3)。
- 用最小值而非均值做中心化,比较与原模型的异同(E4)。
- 把本章 PyMC 代码翻译成 TFP,以及反过来把 TFP 代码翻译成 PyMC,并列出语法差异(E5、E6、
  M14、M15、H17)。
- 用不同强度的先验重新拟合逻辑回归,观察发散情况的变化(E7)。
- 模拟数据并恢复线性模型参数(E9);为已有回归模型生成完整诊断(E10)。
- 把线性回归模型应用到 Gentoo、Chinstrap 企鹅数据上,比较跨物种的后验差异(E11)。
- 反事实分析练习:雌性企鹅、固定喙长条件下的鳍肢长度反事实(M12);协变量重复对 ESS/$\hat R$
  的影响(M13)。
- 用逐渐增加协变量数目的逻辑回归复现"先验预测分布随协变量数目增多而极化"的现象(M16)。
- 修改体重模型使其不能取负值,比较修改前后的先验预测检验与后验(H18)。
- 把喙深、岛屿等 Palmer Penguins 数据集中的其他协变量加入线性/逻辑回归模型,用模型比较工具
  验证是否有帮助(H19、H20)。

## 脚注

[^1]: 更多信息可参阅 TensorFlow 官方教程与文档,例如
    <https://www.tensorflow.org/probability/examples/JointDistributionAutoBatched_A_Gentle_Tutorial>
    与
    <https://www.tensorflow.org/probability/examples/Modeling_with_JointDistribution>。

[^2]: `tfd.Sample` 和 `tfd.Independent` 都是"元分布构造器"——以其他分布为输入、返回一个新
    分布。TFP 中还有其他用途不同的元分布,比如 `tfd.Mixture`、`tfd.TransformedDistribution`
    和 `tfd.JointDistribution`。更全面的 `tfp.distributions` 介绍见
    <https://www.tensorflow.org/probability/examples/TensorFlow_Distributions_Tutorial>。

[^3]: 参见 <https://mc-stan.org/docs/2_23/reference-manual/hmc-algorithm-parameters.html#automatic-parameter-tuning>。

[^4]: 如果想要和原模型完全一致,也可以在 Bambi 里显式指定先验(本文未展示)。但就我们的目的
    而言,这里两个模型已经"足够接近"。

[^5]: 也可以用不同方式解析设计矩阵,让某一列代表两个类别之间的对比。

[^6]: 也许是因为收集更多数据代价高昂、困难重重,甚至根本不可能。

[^7]: 除非是像热带雨林这样的大型系统,植物的存在确实会对气候产生影响——大自然有时候确实很难
    用简单的陈述说清楚。

[^8]: 传统上人们把 $\phi$ 这样的函数应用在等式左边,称之为连接函数(link function)。我们
    则倾向于把它应用在等式右边,为了避免混淆,称之为反连接函数。

[^9]: 传统的广义线性模型文献通常要求观测值的似然属于指数族分布,但贝叶斯方法并不受此限制,
    只要似然能用期望值参数化,原则上都可以使用。

[^10]: 这里的"每单位"指的是吸引力评分每变化一档。
